# File: **energy_balance.csv**

In [ ]:
######################################## Parameters

### Run
name = 'case_transport_1'
prefix = ''


### Threshold to ignore entries
threshold = 1000 # MWh


In [ ]:
##### Import packages
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
warnings.filterwarnings('ignore', category=UserWarning)

Load file and show its content.

In [ ]:
df = xp.load_file_csv(
    params,
    filename='energy_balance.csv',
    location='results',
    prefix=prefix,
    name=name,
    folder='csvs',
    skiprows=4,
    header=None,
    names=['component', 'carrier', 'bus_carrier', 'value'],
)

df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])
df.head()

Remove entries with value below a threshold.

In [ ]:
df_filtered = df[df['value'].abs() >= threshold]

df_filtered.head()

## Waterfall chart

The chart is built in three pieces, all configured through the dictionary `dic` defined right below:

- **`init_generation`** &mdash; first stacked column. Each entry contributes one segment to the *Generation* bar.
- **`intermediate_processes`** &mdash; one waterfall bar per entry. The bar height is the signed sum of the rows selected; positive values raise the running total, negative values lower it. Dashed grey segments connect consecutive levels.
- **`final_loads`** &mdash; last stacked column (values are sign-flipped so consumption is shown as positive). The running total should match this bar's height when the accounting is closed.

Each entry maps a legend name to a *selector* that picks rows from `energy_balance.csv`:

- `(component, carrier, bus_carrier)` &mdash; one specific row.
- A list of such tuples &mdash; sum of the rows it contains (useful to group losses spanning several buses).

Edit `dic` to add, remove or rearrange concepts; the plot adapts automatically.

In [ ]:
#################### Parameters

### Dic to arrange plot
dic = {
    'init_generation': {
        'Onshore wind': ('Generator', 'onwind', 'AC'),
        'Solar rooftop': ('Generator', 'solar rooftop', 'low voltage'),
    },
    'intermediate_processes': {
        'H2 electrolysis': ('Link', 'H2 Electrolysis', 'AC'),
        'H2 turbine': ('Link', 'H2 turbine', 'AC'),        
        'losses (transmission-distribution grid)': [
            ('Link', 'electricity distribution grid', 'AC'),
            ('Link', 'electricity distribution grid', 'low voltage'),
        ],        
        'losses (BEV charger)': [
            ('Link', 'BEV charger', 'low voltage'),
            ('Link', 'BEV charger', 'EV battery'),
        ],
        'V2G': ('Link', 'V2G', 'EV battery'),
    },
    'final_loads': {
        'Land transport EV': ('Load', 'land transport EV', 'EV battery'),
    }
}


### Units
unit_scale = 1e6
unit_label = 'TWh'

### Font sizes
label_fontsize = 14
tick_fontsize = 14
legend_fontsize = 14

In [ ]:
######################################## Helpers

def _resolve_selector(selector):
    """Normalize a selector to a list of (component, carrier, bus_carrier) tuples."""
    # Single-row selector like ('Generator', 'onwind', 'AC') -> wrap in list
    if isinstance(selector, tuple) and selector and isinstance(selector[0], str):
        selector = [selector]
    normalized = []
    for row in selector:
        if len(row) != 3:
            raise ValueError(f"Each row selector must have 3 items (component, carrier, bus_carrier), got {len(row)}")
        normalized.append(tuple(row))
    return normalized


def _selector_value(grouped, selector):
    """Sum values matching a selector against the pre-aggregated table."""
    total = 0.0
    for component, carrier, bus_carrier in _resolve_selector(selector):
        total += grouped.get((component, carrier, bus_carrier), 0.0)
    return total


def _series_from_dict(grouped, d, sign=1):
    return pd.Series(
        {name: sign * _selector_value(grouped, sel) for name, sel in d.items()},
        dtype=float,
    )


######################################## Pre-aggregate (one pass over the dataframe)

grouped = df_filtered.groupby(['component', 'carrier', 'bus_carrier'])['value'].sum()


######################################## Build stages

init_series = _series_from_dict(grouped, dic.get('init_generation', {})) / unit_scale
final_series = _series_from_dict(grouped, dic.get('final_loads', {}), sign=-1) / unit_scale
intermediate_deltas = pd.Series(
    {name: _selector_value(grouped, sel) / unit_scale
     for name, sel in dic.get('intermediate_processes', {}).items()},
    dtype=float,
)

stages = []
if not init_series.empty:
    stages.append({'kind': 'stacked', 'label': 'Generation', 'series': init_series})
for name, delta in intermediate_deltas.items():
    stages.append({'kind': 'delta', 'label': name, 'delta': delta})
if not final_series.empty:
    stages.append({'kind': 'stacked', 'label': 'Loads', 'series': final_series})

if not stages:
    raise ValueError("No data to plot. Check 'dic' configuration.")


######################################## Colors (tech_colors from pypsa-spain config/plotting.default.yaml)

plotting_cfg = xp.load_file_yaml(
    params,
    filename='plotting.default.yaml',
    location='config',
)
tech_colors = plotting_cfg['plotting']['tech_colors']
color_fallback = '#999999'


def _color_for(selector):
    """Resolve a color from tech_colors using the carrier of the first selector row."""
    carrier = _resolve_selector(selector)[0][1]
    return tech_colors.get(carrier, color_fallback)


colors_by_name = {
    name: _color_for(selector)
    for section in ('init_generation', 'intermediate_processes', 'final_loads')
    for name, selector in dic.get(section, {}).items()
}


######################################## Plot helpers

def _plot_stacked(ax, x, series, colors):
    pos_bottom = 0.0
    neg_bottom = 0.0
    for name, value in series.items():
        bottom = pos_bottom if value >= 0 else neg_bottom
        ax.bar(
            x, value, bottom=bottom,
            color=colors[name],
            edgecolor='black', linewidth=0.6,
            label=name,
        )
        if value >= 0:
            pos_bottom += value
        else:
            neg_bottom += value


def _plot_level_link(ax, x, level):
    ax.plot(
        [x - 1, x], [level, level],
        color='gray', linestyle='--', linewidth=0.8,
    )


######################################## Figure

fig, ax = plt.subplots(figsize=(12, 6))

# Reference magnitude to space the numeric labels above/below the delta bars
all_magnitudes = (
    list(init_series.abs().values)
    + list(intermediate_deltas.abs().values)
    + list(final_series.abs().values)
)
label_offset = 0.015 * max(all_magnitudes + [1.0])

current_level = 0.0

for idx, stage in enumerate(stages):

    if stage['kind'] == 'stacked':
        _plot_stacked(ax, idx, stage['series'], colors_by_name)

        if stage['label'] == 'Generation':
            current_level = stage['series'].sum()
        elif idx > 0:
            # Connect waterfall to the final stacked bar
            _plot_level_link(ax, idx, current_level)

    else:  # 'delta'
        delta = stage['delta']
        level_before = current_level
        level_after = current_level + delta

        bar_bottom = min(level_before, level_after)
        bar_height = abs(delta)

        ax.bar(
            idx, bar_height, bottom=bar_bottom,
            color=colors_by_name[stage['label']],
            edgecolor='black', linewidth=0.6,
            label='_nolegend_',
        )

        if idx > 0:
            _plot_level_link(ax, idx, level_before)

        if delta >= 0:
            text_y, text_va = bar_bottom + bar_height + label_offset, 'bottom'
        else:
            text_y, text_va = bar_bottom - label_offset, 'top'
        ax.text(
            idx, text_y, f"{delta:+.2f}",
            ha='center', va=text_va,
            fontsize=max(tick_fontsize - 1, 9),
        )

        current_level = level_after


######################################## Cosmetic

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(range(len(stages)))
ax.set_xticklabels([s['label'] for s in stages], rotation=20, ha='right', fontsize=tick_fontsize)
ax.set_ylabel(f"Energy [{unit_label}]", fontsize=label_fontsize)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)

# Deduplicated legend (skip intermediate bars)
handles, labels = ax.get_legend_handles_labels()
seen = set()
legend_handles, legend_labels = [], []
for h, l in zip(handles, labels):
    if l == '_nolegend_' or l in seen:
        continue
    seen.add(l)
    legend_handles.append(h)
    legend_labels.append(l)

ax.legend(
    legend_handles, legend_labels,
    loc='upper left', bbox_to_anchor=(1.01, 1),
    frameon=False, fontsize=legend_fontsize,
)

plt.tight_layout()
plt.show()
